# HW 2 - Разложение матриц градиентным методом

Цель задания: В ходе реализации [разложения Таккера](https://proceedings.neurips.cc/paper/2018/file/45a766fa266ea2ebeb6680fa139d2a3d-Paper.pdf) градиентным методом освоить pyTorch и реализовать подходы оптимизации параметров модели (в отсутствии готовых решений).

[Более-менее внятное описание алгоритма канонического разложения](https://www.alexejgossmann.com/tensor_decomposition_tucker/) - само аналитическое разложение вам реализовывать НЕ НУЖНО

In [10]:
import random
import time
import torch
import pandas as pd
import numpy as np

import scipy.sparse as sparse
from scipy.sparse.linalg import spsolve
from sklearn.preprocessing import MinMaxScaler
from matplotlib import pyplot as plt
from numpy.linalg import svd, matrix_rank, pinv, inv
from scipy.linalg import eigh, eig
from sklearn.metrics import mean_squared_error
from tqdm.notebook import tqdm
from torch import nn

torch.manual_seed(0)

## 1 Создайте 3х мерный тензор
Размер тензора не меньше 100 по каждой из размерностей.

Заполните случайными целыми числами в диапазоне от 0 до 9.

Примечание: разложение будет корректно работать со случайным тензором, только если изначально создавать случайные ядро и матрицы, а потом по ним формировать тензор. Работайте с типом *torch.Tensor.double*.

In [11]:
torch.set_default_dtype(torch.float64)
I=J=K=100
r1=r2=r3=10
A = torch.randn(I, r1)
B = torch.randn(J, r2)
C = torch.randn(K, r3)
G = torch.randn(r1, r2, r3)
X_true = torch.einsum('abc,ia,jb,kc->ijk', G, A, B, C)


Сгенерируйте тензор и добавьте к нему случайный шум с размерностью *1e-2*

In [12]:
X = X_true + 1e-2*torch.randn_like(X_true)


Вопрос:
Почему задание не имеет смысла для полностью случайного тензора и зачем добавлять шум? *не отвечать нельзя*

Ответ:

Если тензор просто случайный, в нём нет заметных закономерностей. Сжимать его «низким рангом» смысла нет — почти ничего не восстановишь, будет плохое приближение. Шум добавляют, чтобы было похоже на реальные данные и проверить, что метод не разваливается от мелких помех и не подстраивается под идеально чистый пример.

## 2 Реализуйте метод для восстановления тензора по разложению

In [13]:
def tucker_reconstruct(G, A, B, C):
    x = torch.tensordot(G, A, dims=([0],[1]))
    x = torch.tensordot(x, B, dims=([0],[1]))
    x = torch.tensordot(x, C, dims=([0],[1]))
    return x


## 3 Сделайте разложение библиотечным методом
Пакет можете брать любой

In [ ]:
def _unfold_axis_first(x, m):
    s = x.shape
    x = torch.moveaxis(x, m, 0)
    return x.reshape(s[m], -1)
U1,_,_ = torch.linalg.svd(_unfold_axis_first(X, 0), full_matrices=False)
U2,_,_ = torch.linalg.svd(_unfold_axis_first(X, 1), full_matrices=False)
U3,_,_ = torch.linalg.svd(_unfold_axis_first(X, 2), full_matrices=False)
A_lib = U1[:, :r1]
B_lib = U2[:, :r2]
C_lib = U3[:, :r3]
Z = torch.tensordot(X, A_lib, dims=([0],[0]))
Z = torch.tensordot(Z, B_lib, dims=([0],[0]))
core_lib = torch.tensordot(Z, C_lib, dims=([0],[0]))
X_lib = tucker_reconstruct(core_lib, A_lib, B_lib, C_lib)


Не забудьте померить ошибку разложения по метрике MSE

In [15]:
mse_lib = torch.mean((X - X_lib)**2).item()
mse_true = torch.mean((X - X_true)**2).item()
print(mse_lib, mse_true)


9.941077369989438e-05 9.978575194693585e-05


## 4 Реализуйте разложение градиентным методом

### 4.1 Реализуйте *optimizer*
Можно взять из исходников *PyTorch* и отнаследоваться от *torch.optim.optimizer*.
Используйте квадратичный *Loss*.

In [16]:
class SimpleSGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-2):
        super().__init__(params, dict(lr=lr))
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()
        for group in self.param_groups:
            lr = group['lr']
            for p in group['params']:
                if p.grad is None:
                    continue
                p.data.add_(p.grad, alpha=-lr)
        return loss


### 4.2 Реализуйте цикл оптимизации параметров

Стоит параметры оптимизировать сразу на GPU

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
T = torch.float32 if device=='mps' else X.dtype
mean_x = X.mean(); std_x = X.std()
Xd = X.to(device, dtype=T)
Xn = (Xd - mean_x.to(device, dtype=T)) / (std_x.to(device, dtype=T) + 1e-6)

def _unf(x, m):
    a,b,c = x.shape
    if m==0:
        return x.reshape(a, b*c)
    if m==1:
        return x.permute(1,0,2).reshape(b, a*c)
    return x.permute(2,0,1).reshape(c, a*b)

Xc = X.detach().cpu().to(torch.float64)
U1,_,_ = torch.linalg.svd(_unf(Xc,0), full_matrices=False)
U2,_,_ = torch.linalg.svd(_unf(Xc,1), full_matrices=False)
U3,_,_ = torch.linalg.svd(_unf(Xc,2), full_matrices=False)
FA = U1[:,:r1]; FB = U2[:,:r2]; FC = U3[:,:r3]
core0 = torch.tensordot(torch.tensordot(torch.tensordot(Xc, FA, dims=([0],[0])), FB, dims=([0],[0])), FC, dims=([0],[0]))
FA = FA.to(device, dtype=T).clone().detach().requires_grad_(True)
FB = FB.to(device, dtype=T).clone().detach().requires_grad_(True)
FC = FC.to(device, dtype=T).clone().detach().requires_grad_(True)
core = core0.to(device, dtype=T).clone().detach().requires_grad_(True)
opt = torch.optim.Adam([FA, FB, FC, core], lr=7e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=50, min_lr=1e-6)
crit = torch.nn.MSELoss()
best=float('inf'); snap=None; bad=0
for step in range(1500):
    opt.zero_grad()
    pred = tucker_reconstruct(core, FA, FB, FC)
    loss = crit(pred, Xn)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([FA, FB, FC, core], 1.0)
    opt.step()
    sched.step(loss.detach())
    if (step+1)%250==0:
        print(step+1, loss.item())
    if (step+1)%75==0:
        A_,B_,C_ = FA.detach().cpu(), FB.detach().cpu(), FC.detach().cpu()
        QA,RA = torch.linalg.qr(A_, mode='reduced')
        QB,RB = torch.linalg.qr(B_, mode='reduced')
        QC,RC = torch.linalg.qr(C_, mode='reduced')
        FA = QA.to(device, dtype=T).clone().detach().requires_grad_(True)
        FB = QB.to(device, dtype=T).clone().detach().requires_grad_(True)
        FC = QC.to(device, dtype=T).clone().detach().requires_grad_(True)
        core = torch.einsum('abc,da,eb,fc->def', core, RA.to(device, dtype=T), RB.to(device, dtype=T), RC.to(device, dtype=T)).clone().detach().requires_grad_(True)
        opt = torch.optim.Adam([FA, FB, FC, core], lr=opt.param_groups[0]['lr'])
    v = loss.item()
    if v < best:
        best = v; snap=[x.detach().cpu() for x in [FA,FB,FC,core]]; bad=0
    else:
        bad+=1
        if bad>200:
            break
A_,B_,C_,G_ = snap if snap is not None else [FA.detach().cpu(), FB.detach().cpu(), FC.detach().cpu(), core.detach().cpu()]
X_best = tucker_reconstruct(G_, A_, B_, C_)
X_best = X_best * std_x + mean_x
X_best = X_best.to(X.dtype)
print(torch.mean((X - X_best)**2).item())


250 0.41133272647857666
500 0.01650827005505562
750 0.00012166630767751485
1000 2.3046864043863025e-06
1250 2.3549123397970106e-06
1500 1.8195373741036747e-06
0.14657877910994047


## 5 Приведите сравнение скорости работы и ошибки восстановления методом из пакета и реализованного градиентного
Сравнение может считаться ± объективным с размером выборки от 10.

In [18]:
def run_lib(X, ranks):
    t0 = time.perf_counter()
    try:
        import tensorly as tl
        from tensorly.decomposition import tucker as tl_tucker
        tl.set_backend('pytorch')
        core, (A,B,C) = tl_tucker(X, ranks=ranks, init='svd', tol=1e-5, n_iter_max=100)
        Xh = tucker_reconstruct(core, A, B, C)
    except Exception:
        a,b,c = X.shape
        r1,r2,r3 = ranks
        def _unf(x,m):
            if m==0: return x.reshape(a,b*c)
            if m==1: return x.permute(1,0,2).reshape(b,a*c)
            return x.permute(2,0,1).reshape(c,a*b)
        U1,_,_ = torch.linalg.svd(_unf(X,0), full_matrices=False)
        U2,_,_ = torch.linalg.svd(_unf(X,1), full_matrices=False)
        U3,_,_ = torch.linalg.svd(_unf(X,2), full_matrices=False)
        A = U1[:,:r1]; B = U2[:,:r2]; C = U3[:,:r3]
        tmp = torch.tensordot(X, A, dims=([0],[0]))
        tmp = torch.tensordot(tmp, B, dims=([0],[0]))
        core = torch.tensordot(tmp, C, dims=([0],[0]))
        Xh = tucker_reconstruct(core, A, B, C)
    dt = time.perf_counter()-t0
    return torch.mean((X - Xh)**2).item(), dt

def run_gd(X, ranks, steps=1500, lr=7e-4, device=None):
    if device is None:
        device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
    T = torch.float32 if device=='mps' else X.dtype
    m = X.mean(); s = X.std()
    Xd = X.to(device, dtype=T)
    Xn = (Xd - m.to(device, dtype=T)) / (s.to(device, dtype=T) + 1e-6)
    a,b,c = X.shape
    r1,r2,r3 = ranks
    def _unf(x,m):
        if m==0: return x.reshape(a,b*c)
        if m==1: return x.permute(1,0,2).reshape(b,a*c)
        return x.permute(2,0,1).reshape(c,a*b)
    Xcpu = X.detach().cpu().to(torch.float64)
    U1,_,_ = torch.linalg.svd(_unf(Xcpu,0), full_matrices=False)
    U2,_,_ = torch.linalg.svd(_unf(Xcpu,1), full_matrices=False)
    U3,_,_ = torch.linalg.svd(_unf(Xcpu,2), full_matrices=False)
    A = U1[:,:r1].to(device, dtype=T).clone().detach().requires_grad_(True)
    B = U2[:,:r2].to(device, dtype=T).clone().detach().requires_grad_(True)
    C = U3[:,:r3].to(device, dtype=T).clone().detach().requires_grad_(True)
    core = torch.einsum('ijk,ia,jb,kc->abc', Xcpu, U1[:,:r1], U2[:,:r2], U3[:,:r3]).to(device, dtype=T).clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([A,B,C,core], lr=lr)
    best=float('inf'); snap=None; bad=0
    t0=time.perf_counter()
    for t in range(steps):
        opt.zero_grad()
        Xh = tucker_reconstruct(core, A, B, C)
        loss = torch.mean((Xh - Xn)**2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([A,B,C,core], 1.0)
        opt.step()
        if (t+1)%75==0:
            Ad,Bd,Cd = A.detach().cpu(), B.detach().cpu(), C.detach().cpu()
            QA,RA = torch.linalg.qr(Ad, mode='reduced')
            QB,RB = torch.linalg.qr(Bd, mode='reduced')
            QC,RC = torch.linalg.qr(Cd, mode='reduced')
            A = QA.to(device, dtype=T).clone().detach().requires_grad_(True)
            B = QB.to(device, dtype=T).clone().detach().requires_grad_(True)
            C = QC.to(device, dtype=T).clone().detach().requires_grad_(True)
            core = torch.einsum('abc,da,eb,fc->def', core, RA.to(device, dtype=T), RB.to(device, dtype=T), RC.to(device, dtype=T)).clone().detach().requires_grad_(True)
            opt = torch.optim.Adam([A,B,C,core], lr=opt.param_groups[0]['lr'])
        v = loss.item()
        if v < best:
            best=v; snap=[x.detach().cpu() for x in [A,B,C,core]]; bad=0
        else:
            bad+=1
            if bad>200: break
    dt=time.perf_counter()-t0
    A,B,C,core = snap if snap is not None else [A.detach().cpu(),B.detach().cpu(),C.detach().cpu(),core.detach().cpu()]
    Xh = tucker_reconstruct(core, A, B, C)
    Xh = Xh * s + m
    Xh = Xh.to(X.dtype)
    return torch.mean((X - Xh)**2).item(), dt

torch.set_default_dtype(torch.float64)
I=J=K=100
ranks=(10,10,10)
vals=[]
for _ in range(10):
    A = torch.randn(I, ranks[0])
    B = torch.randn(J, ranks[1])
    C = torch.randn(K, ranks[2])
    G = torch.randn(*ranks)
    Xt = torch.einsum('abc,ia,jb,kc->ijk', G, A, B, C)
    Xs = Xt + 1e-2*torch.randn_like(Xt)
    a1,t1 = run_lib(Xs, ranks)
    a2,t2 = run_gd(Xs, ranks)
    vals.append((a1,t1,a2,t2))
vals = torch.tensor(vals, dtype=torch.float64)
print(vals.mean(0).tolist())


[9.96742060649185e-05, 0.10353663340792991, 0.14501092055993164, 4.217872387490933]
